In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr
from scipy.stats import chisquare
import warnings
warnings.filterwarnings('ignore')

# Suppress matplotlib font warnings
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Load data
df = pd.read_csv('dataset_v3.csv')

# Screen_Time_vs_Mental_Health

In [3]:
from scipy.stats import chi2_contingency
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from datetime import datetime

# Fixed margins for PDF output
LEFT_MARGIN = 0.10
RIGHT_MARGIN = 0.90

def analysis(df, target, variables, section_title, save_image=True, filename="analysis_results.pdf"):
  
    results = []
    crosstabs = []
    
    for i, var in enumerate(variables, 1):
        ct = pd.crosstab(df[var], df[target], margins=True, margins_name='Total')
        crosstabs.append((var, ct))
        
        ct_test = pd.crosstab(df[var], df[target])
        chi2, p, dof, expected = chi2_contingency(ct_test)
        
        significance = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

        results.append({
            'Variable': var,
            'χ² Statistic': round(chi2, 3),
            'df': dof,
            'p-value': round(p, 4),
            'Sig.': significance
        })

    chi_square_table = pd.DataFrame(results)
    
    # Save all tables as multi-page A4 PDF with clean professional styling
    if save_image:
        plt.rcParams['font.family'] = 'DejaVu Sans'
        plt.rcParams['font.size'] = 10
        
        fig_width = 8.5
        fig_height = 11
        table_width = RIGHT_MARGIN - LEFT_MARGIN
        
        with PdfPages(filename) as pdf:
            page_num = 1
            
            # Page 1: Title/Header
            fig = plt.figure(figsize=(fig_width, fig_height))
            fig.patch.set_facecolor('white')
            ax = fig.add_subplot(111)
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            ax.text(0.5, 0.90, section_title, transform=ax.transAxes, 
                   fontsize=16, fontweight='bold', fontfamily='DejaVu Sans', 
                   va='top', ha='center', color='black')
            
            ax.text(LEFT_MARGIN, 0.80, "Outcome Variable:", transform=ax.transAxes,
                   fontsize=11, fontweight='bold', fontfamily='DejaVu Sans', va='top')
            ax.text(LEFT_MARGIN + 0.20, 0.80, target, transform=ax.transAxes,
                   fontsize=11, fontfamily='DejaVu Sans', va='top')
            
            ax.text(LEFT_MARGIN, 0.70, "Predictor Variables:", transform=ax.transAxes,
                   fontsize=11, fontweight='bold', fontfamily='DejaVu Sans', va='top')
            
            var_display = '\n'.join(variables)
            ax.text(LEFT_MARGIN, 0.68, var_display, transform=ax.transAxes,
                   fontsize=9, fontfamily='DejaVu Sans', va='top')
            
            ax.text(0.5, 0.03, f'Page {page_num}', transform=ax.transAxes, 
                   fontsize=9, fontfamily='DejaVu Sans', ha='center')
            page_num += 1
            
            pdf.savefig(fig, bbox_inches='tight', facecolor='white', dpi=300)
            plt.close()
            
            # Pages 2+: Individual tables
            for var, ct in crosstabs:
                fig = plt.figure(figsize=(fig_width, fig_height))
                fig.patch.set_facecolor('white')
                ax = fig.add_subplot(111)
                ax.axis('off')
                for spine in ax.spines.values():
                    spine.set_visible(False)
                
                ax.text(0.5, 0.96, f"Cross-tabulation: {var} × {target}", 
                       transform=ax.transAxes, fontsize=12, fontweight='bold', 
                       fontfamily='DejaVu Sans', ha='center', va='top')
                
                row_labels = list(ct.index)
                col_labels = list(ct.columns)
                cell_data = []
                
                for row_label in row_labels:
                    row_idx = list(ct.index).index(row_label)
                    row_data = [str(int(val)) for val in ct.values[row_idx]]
                    cell_data.append(row_data)
                
                full_data = []
                for i, label in enumerate(row_labels):
                    full_data.append([str(label)] + cell_data[i])
                
                all_col_labels = [var] + [str(col) for col in col_labels]
                
                table = ax.table(
                    cellText=full_data,
                    colLabels=all_col_labels,
                    cellLoc='center',
                    loc='upper center',
                    bbox=[0.15, 0.20, 0.70, 0.65]
                )
                table.auto_set_font_size(False)
                table.set_fontsize(8)
                
                for (row, col), cell in table.get_celld().items():
                    cell.set_text_props(fontfamily='DejaVu Sans', fontsize=9)
                    cell.set_facecolor('white')
                    cell.set_edgecolor('none')
                    cell.set_linewidth(0)
                    
                    if row == 0:
                        cell.set_facecolor('#f5f5f5')
                        cell.set_text_props(fontweight='bold', fontfamily='DejaVu Sans', fontsize=9, rotation=90)
                        cell.set_height(0.08)
                
                ax.text(0.92, 0.02, f'Page {page_num}', transform=ax.transAxes, 
                       fontsize=9, fontfamily='DejaVu Sans', ha='right')
                page_num += 1
                
                pdf.savefig(fig, bbox_inches='tight', facecolor='white', dpi=300)
                plt.close()
            
            # Final page: Chi-square summary
            fig = plt.figure(figsize=(fig_width, fig_height))
            fig.patch.set_facecolor('white')
            ax = fig.add_subplot(111)
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            ax.text(0.5, 0.96, f"Statistical Summary: Chi-Square Test Results", 
                   transform=ax.transAxes, fontsize=12, fontweight='bold', 
                   fontfamily='DejaVu Sans', ha='center', va='top')
            
            chi_df = chi_square_table
            table = ax.table(
                cellText=chi_df.values,
                colLabels=chi_df.columns,
                cellLoc='center',
                loc='upper center',
                bbox=[LEFT_MARGIN + 0.05, 0.20, 0.80, 0.70]
            )
            table.auto_set_font_size(False)
            table.set_fontsize(8)
            
            for (row, col), cell in table.get_celld().items():
                cell.set_text_props(fontfamily='DejaVu Sans', fontsize=9)
                cell.set_facecolor('white')
                cell.set_edgecolor('none')
                cell.set_linewidth(0)
                
                if row == 0:
                    cell.set_facecolor('#f5f5f5')
                    cell.set_text_props(fontweight='bold', fontfamily='DejaVu Sans', fontsize=9)
            
            ax.text(0.08, 0.10, "Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant",
                   transform=ax.transAxes, fontsize=8, fontfamily='DejaVu Sans', va='top')
            
            ax.text(0.92, 0.02, f'Page {page_num}', transform=ax.transAxes, 
                   fontsize=9, fontfamily='DejaVu Sans', ha='right')
            
            pdf.savefig(fig, bbox_inches='tight', facecolor='white', dpi=300)
            plt.close()
        
        plt.rcParams['font.family'] = plt.rcParamsDefault['font.family']
        print(f"✓ Report saved: {filename}")
    
    return chi_square_table

target = 'D1_daily_screen'
variables = ['PHQ_class','G1_mood_swings', 'G2_anxious_without_device',
          'B5_time_with_parents_hours', 'B6_relationship_with_family',
          'D3_device', 'G5_negative_mental', 'G6_panic_attack']


_ = analysis(df, target, variables, section_title="Screen_Time_vs_Mental_Health", filename="report_v4/1.Screen_Time_vs_Mental_Health.pdf")

✓ Report saved: report_v4/1.Screen_Time_vs_Mental_Health.pdf


# Screen Time VS Physical Health

In [4]:
target = 'D1_daily_screen'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Screen_Time_vs_Physical_Health", filename="report_v4/2.Screen_Time_vs_Physical_Health.pdf")

✓ Report saved: report_v4/2.Screen_Time_vs_Physical_Health.pdf


# Weekend Screen Time VS Physical Health

In [5]:
target = 'D2_weekend_screen'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Weekend_Screen_Time_vs_Physical_Health", filename="report_v4/3.Weekend_Screen_Time_vs_Physical_Health.pdf")

✓ Report saved: report_v4/3.Weekend_Screen_Time_vs_Physical_Health.pdf


# Weekend Screen Time VS Mental Health

In [6]:
target = 'D2_weekend_screen'
variables = ['PHQ_class','G1_mood_swings', 'G2_anxious_without_device',
          'B5_time_with_parents_hours', 'B6_relationship_with_family',
          'D3_device', 'G5_negative_mental', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Weekend_Screen_Time_vs_Mental_Health", filename="report_v4/4.Weekend_Screen_Time_vs_Mental_Health.pdf")

✓ Report saved: report_v4/4.Weekend_Screen_Time_vs_Mental_Health.pdf


# Primary Device Used VS Physical Health

In [7]:
target = 'D3_device'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Primary_Device_Used_vs_Physical_Health", filename="report_v4/5.Primary_Device_Used_vs_Physical_Health.pdf")

✓ Report saved: report_v4/5.Primary_Device_Used_vs_Physical_Health.pdf


# Primary Device Used VS Mental Health

In [8]:
target = 'D3_device'
variables = ['PHQ_class','G1_mood_swings', 'G2_anxious_without_device',
          'B5_time_with_parents_hours', 'B6_relationship_with_family',
          'D1_daily_screen', 'G5_negative_mental', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Primary_Device_Used_vs_Mental_Health", filename="report_v4/6.Primary_Device_Used_vs_Mental_Health.pdf")

✓ Report saved: report_v4/6.Primary_Device_Used_vs_Mental_Health.pdf


# Sleep Duration VS Physical Health

In [9]:
target = 'E1_sleep_duration'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'D1_daily_screen', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Sleep_Duration_vs_Physical_Health", filename="report_v4/7.Sleep_Duration_vs_Physical_Health.pdf")

✓ Report saved: report_v4/7.Sleep_Duration_vs_Physical_Health.pdf


# Sleep Duration VS Mental Health

In [10]:
target = 'E1_sleep_duration'
variables = ['PHQ_class','G1_mood_swings', 'G2_anxious_without_device',
          'B5_time_with_parents_hours', 'B6_relationship_with_family',
          'D1_daily_screen', 'G5_negative_mental', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Sleep_Duration_vs_Mental_Health", filename="report_v4/8.Sleep_Duration_vs_Mental_Health.pdf")

✓ Report saved: report_v4/8.Sleep_Duration_vs_Mental_Health.pdf


# Usual Bedtime VS Physical Health

In [11]:
target = 'E5_bedtime'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'D1_daily_screen',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Usual_Bedtime_vs_Physical_Health", filename="report_v4/9.Usual_Bedtime_vs_Physical_Health.pdf")

✓ Report saved: report_v4/9.Usual_Bedtime_vs_Physical_Health.pdf


# Usual Bedtime VS Mental Health

In [12]:
target = 'E5_bedtime'
variables = ['PHQ_class', 'G1_mood_swings', 'G2_anxious_without_device',
             'B5_time_with_parents_hours', 'B6_relationship_with_family',
             'D1_daily_screen', 'G5_negative_mental', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Usual_Bedtime_vs_Mental_Health", filename="report_v4/10.Usual_Bedtime_vs_Mental_Health.pdf")

✓ Report saved: report_v4/10.Usual_Bedtime_vs_Mental_Health.pdf


# Anxious Without Device VS Physical Health

In [13]:
target = 'G2_anxious_without_device'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Anxious_Without_Device_vs_Physical_Health", filename="report_v4/11.Anxious_Without_Device_vs_Physical_Health.pdf")

✓ Report saved: report_v4/11.Anxious_Without_Device_vs_Physical_Health.pdf


# Anxious Without Device VS Mental Health

In [14]:
target = 'G2_anxious_without_device'
variables = ['PHQ_class', 'G1_mood_swings', 'B5_time_with_parents_hours',
             'B6_relationship_with_family', 'D1_daily_screen',
             'D3_device', 'G5_negative_mental', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Anxious_Without_Device_vs_Mental_Health", filename="report_v4/12.Anxious_Without_Device_vs_Mental_Health.pdf")

✓ Report saved: report_v4/12.Anxious_Without_Device_vs_Mental_Health.pdf


# Feel Low or Unmotivated VS Physical Health

In [15]:
target = 'G5_negative_mental'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Feel_Low_Unmotivated_vs_Physical_Health", filename="report_v4/13.Feel_Low_Unmotivated_vs_Physical_Health.pdf")

✓ Report saved: report_v4/13.Feel_Low_Unmotivated_vs_Physical_Health.pdf


# Feel Low or Unmotivated VS Mental Health

In [16]:
target = 'G5_negative_mental'
variables = ['PHQ_class', 'G1_mood_swings', 'G2_anxious_without_device',
             'B5_time_with_parents_hours', 'B6_relationship_with_family',
             'D1_daily_screen', 'D3_device', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Feel_Low_Unmotivated_vs_Mental_Health", filename="report_v4/14.Feel_Low_Unmotivated_vs_Mental_Health.pdf")

✓ Report saved: report_v4/14.Feel_Low_Unmotivated_vs_Mental_Health.pdf


# Academic Satisfaction VS Physical Health

In [17]:
target = 'F1_academic_satisfaction'
variables = ['weight_class', 'BMI_class', 'C3_physical_problems',
             'E1_sleep_duration', 'E3_sleep_medicine', 'E5_bedtime',
             'E6_wake_time', 'H2_appetite_change']

_ = analysis(df, target, variables, section_title="Academic_Satisfaction_vs_Physical_Health", filename="report_v4/15.Academic_Satisfaction_vs_Physical_Health.pdf")

✓ Report saved: report_v4/15.Academic_Satisfaction_vs_Physical_Health.pdf


# Academic Satisfaction VS Mental Health

In [18]:
target = 'F1_academic_satisfaction'
variables = ['PHQ_class', 'G1_mood_swings', 'G2_anxious_without_device',
             'B5_time_with_parents_hours', 'B6_relationship_with_family',
             'D1_daily_screen', 'G5_negative_mental', 'G6_panic_attack']

_ = analysis(df, target, variables, section_title="Academic_Satisfaction_vs_Mental_Health", filename="report_v4/16.Academic_Satisfaction_vs_Mental_Health.pdf")

✓ Report saved: report_v4/16.Academic_Satisfaction_vs_Mental_Health.pdf
